# OpenWEC — WEC Season Comparison

This notebook compares WEC seasons (2024, 2025, 2026) across:
- Race and lap counts per season
- Manufacturer presence and wins
- HYPERCAR pace evolution
- Class breakdown

**Requirements:**
```bash
pip install openwec[plotting] httpx
```

Most data here uses public endpoints — no API key needed for stats and results.
Pace analysis requires an API key.

## 1. Setup

In [ ]:
import openwec
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import httpx
import os

API_KEY  = os.environ.get("OPENWEC_API_KEY", "")
BASE_URL = "https://api.openwec.com/api/v1"

openwec.configure(api_key=API_KEY)
headers = {"X-API-Key": API_KEY} if API_KEY else {}
client  = httpx.Client(base_url=BASE_URL, headers=headers, timeout=30)

SEASONS  = [2024, 2025, 2026]
SERIES   = "WEC"
COLORS   = {2024: '#4895EF', 2025: '#6FCF97', 2026: '#FFB000'}

print(f"openwec {openwec.__version__}")

## 2. Season overview stats

In [ ]:
stats = {}
for year in SEASONS:
    r = client.get(f"/series/{SERIES}/seasons/{year}/stats")
    if r.status_code == 200:
        stats[year] = r.json()
        print(f"{year}: {stats[year]['total_races']} races, "
              f"{stats[year]['total_laps']:,} laps, "
              f"{len(stats[year]['manufacturers'])} manufacturers")
    else:
        print(f"{year}: not available ({r.status_code})")

In [ ]:
# Season overview table
overview = pd.DataFrame([
    {
        "Season": year,
        "Races": s["total_races"],
        "Laps": s["total_laps"],
        "Entries": s["total_entries"],
        "Manufacturers": len(s["manufacturers"]),
        "Classes": len([c for c in s["classes"] if c in ["HYPERCAR", "LMP2", "LMGT3"]]),
    }
    for year, s in stats.items()
])
overview

## 3. Manufacturer presence across seasons

In [ ]:
# Which manufacturers competed in each season?
all_manufacturers = set()
for s in stats.values():
    all_manufacturers.update(s["manufacturers"])

manufacturer_df = pd.DataFrame(
    {year: {m: m in s["manufacturers"] for m in all_manufacturers}
     for year, s in stats.items()}
).T

print("Manufacturer presence (True = competed):")
print(manufacturer_df.to_string())

In [ ]:
# Heatmap-style visualization
fig, ax = plt.subplots(figsize=(14, 4))
fig.patch.set_facecolor("#14181F")
ax.set_facecolor("#1C222C")

manufacturers_sorted = sorted(all_manufacturers)
for i, year in enumerate(SEASONS):
    for j, mfr in enumerate(manufacturers_sorted):
        present = mfr in stats[year]["manufacturers"]
        ax.add_patch(plt.Rectangle(
            (j - 0.4, i - 0.35), 0.8, 0.7,
            color=COLORS[year] if present else '#2A313F',
            alpha=0.85 if present else 1.0
        ))

ax.set_xlim(-0.5, len(manufacturers_sorted) - 0.5)
ax.set_ylim(-0.5, len(SEASONS) - 0.5)
ax.set_xticks(range(len(manufacturers_sorted)))
ax.set_xticklabels(manufacturers_sorted, rotation=45, ha='right', color='#8B95A7', fontsize=10)
ax.set_yticks(range(len(SEASONS)))
ax.set_yticklabels(SEASONS, color='#8B95A7')
ax.set_title("WEC Manufacturer Presence by Season", color='#ECEFF4', pad=12)
ax.tick_params(length=0)

for i, (year, color) in enumerate(COLORS.items()):
    ax.text(-0.6, i, str(year), color=color, fontfamily='monospace',
            fontsize=10, fontweight='bold', ha='right', va='center')

plt.tight_layout()
plt.show()

## 4. Top drivers by season

In [ ]:
for year, s in stats.items():
    print(f"\n{year} — Top drivers by wins:")
    for d in s["top_drivers"][:5]:
        country = f"({d['country']})" if d["country"] else ""
        print(f"  {d['first_name']} {d['last_name']} {country}: {d['wins']} win(s)")

## 5. HYPERCAR pace comparison across seasons

Compare average HYPERCAR pace at Le Mans across seasons.
Requires API key.

In [ ]:
if not API_KEY:
    print("API key required for pace data.")
    print("Request one at https://openwec.com/api-keys")
else:
    le_mans_pace = {}

    for year in SEASONS:
        try:
            session = openwec.Session(SERIES, year, "Le Mans", "Race")
            pace = session.pace(car_class="HYPERCAR")
            if not pace.empty:
                le_mans_pace[year] = pace
                median = pace["avg_pace_s"].median()
                best   = pace["avg_pace_s"].min()
                print(f"{year} Le Mans — HYPERCAR pace: median {median:.3f}s, best {best:.3f}s")
        except Exception as e:
            print(f"{year}: {e}")

In [ ]:
if API_KEY and le_mans_pace:
    fig, ax = plt.subplots(figsize=(10, 5))
    fig.patch.set_facecolor("#14181F")
    ax.set_facecolor("#1C222C")

    for year, pace in le_mans_pace.items():
        clean = pace["avg_pace_s"].dropna().sort_values()
        ax.plot(
            range(len(clean)), clean.values,
            marker='o', markersize=4,
            label=str(year),
            color=COLORS[year],
            linewidth=1.5
        )

    def fmt_lap(s, pos=None):
        if s is None: return ''
        m = int(s // 60)
        sec = s % 60
        return f"{m}:{sec:06.3f}"

    ax.yaxis.set_major_formatter(ticker.FuncFormatter(fmt_lap))
    ax.set_xlabel("Car rank (fastest → slowest)", color='#8B95A7')
    ax.set_ylabel("Avg green-flag lap time", color='#8B95A7')
    ax.set_title("WEC Le Mans — HYPERCAR Pace Comparison", color='#ECEFF4')
    ax.tick_params(colors='#8B95A7')
    ax.legend(facecolor='#232A36', labelcolor='#ECEFF4')
    ax.grid(alpha=0.2)

    plt.tight_layout()
    plt.show()

## 6. Lap count evolution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor("#14181F")

for ax in axes:
    ax.set_facecolor("#1C222C")
    ax.tick_params(colors='#8B95A7')

years  = list(stats.keys())
laps   = [stats[y]["total_laps"] for y in years]
races  = [stats[y]["total_races"] for y in years]
colors = [COLORS[y] for y in years]

axes[0].bar(years, laps, color=colors, edgecolor='#14181F')
axes[0].set_title("Total Laps per Season", color='#ECEFF4')
axes[0].set_ylabel("Laps", color='#8B95A7')

axes[1].bar(years, races, color=colors, edgecolor='#14181F')
axes[1].set_title("Total Races per Season", color='#ECEFF4')
axes[1].set_ylabel("Races", color='#8B95A7')

for ax in axes:
    ax.set_xticks(years)

plt.tight_layout()
plt.show()

---

## Next steps

- [le_mans_2026.ipynb](le_mans_2026.ipynb) — deep dive into Le Mans 2026
- [driver_career.ipynb](driver_career.ipynb) — cross-series career analysis
- [API Documentation](https://api.openwec.com/docs)
- [Request API key](https://openwec.com/api-keys)